# Optimisation de la valeur de stock — Horizon restant de l'année fiscale

## 1. Contexte et problématique

Nous sommes en **mai** : il reste **5 mois** avant la clôture de l'année fiscale (mai à
septembre). Un état des lieux du stock fait apparaître une valeur totale actuelle d'environ
**4 000 000 €**, très au-dessus de la **cible fiscale de fin d'année, fixée à 2 000 000 €**.

L'objectif est de déterminer, pour chaque référence, **quand et combien commander** (ou
s'abstenir de commander) sur les 5 mois restants, de façon à :
- satisfaire la demande client à chaque période,
- respecter un stock de sécurité minimal (à partir du 2ᵉ mois),
- ramener la valeur totale du stock en fin d'année **à 2 000 000 € ou moins**,
- tenir compte de deux contraintes opérationnelles supplémentaires :
  - **la fermeture des fournisseurs en août** (aucune commande ne peut être passée ce mois-là,
    même si elle peut être reçue si elle a été passée avant),
  - **une capacité de stockage limitée** (en m³).

## 2. Formulation mathématique

### Ensembles
| Symbole | Signification |
|---|---|
| $I$ | ensemble des références produit |
| $T = \{1,\dots,5\}$ | périodes (1=Mai, 2=Juin, 3=Juillet, 4=Août, 5=Septembre) |
| $A \subset T$ | périodes de fermeture fournisseur ($A = \{4\}$, Août) |

### Paramètres
| Symbole | Signification |
|---|---|
| $p_i$ | prix unitaire de la référence $i$ |
| $S^0_i$ | stock initial (mai) |
| $d_{i,t}$ | demande de la référence $i$ en période $t$ |
| $LT_i$ | délai de livraison (en mois) |
| $MOQ_i$ | quantité minimale de commande |
| $v_i$ | volume unitaire de stockage (m³) |
| $SS_i$ | stock de sécurité |
| $c^s_i$ | coût de possession unitaire par mois |
| $c^o_i$ | coût fixe de passation d'une commande |
| $TARGET$ | cible de valeur de stock en fin d'horizon |
| $V_{max}$ | capacité totale de stockage (m³) |
| $M$ | grand-M (borne supérieure large sur les commandes) |

### Variables de décision
| Symbole | Type | Signification |
|---|---|---|
| $S_{i,t} \ge 0$ | continue | stock de $i$ en fin de période $t$ |
| $O_{i,t} \ge 0$ | continue | quantité commandée de $i$ en période $t$ |
| $z_{i,t} \in \{0,1\}$ | binaire | 1 si une commande de $i$ est passée en $t$ |

### Fonction objectif

Minimiser le coût total de possession et de passation de commandes :
$$
\min \quad Z = \sum_{i \in I}\sum_{t \in T} c^s_i \, S_{i,t} \;+\; \sum_{i \in I}\sum_{t \in T} c^o_i \, z_{i,t}
$$

### Contraintes

**C1 — Bilan de stock** (le stock initial $S^0_i$ intervient pour $t=1$) :
$$
S_{i,t} = S_{i,t-1} + O_{i,\,t-LT_i}\cdot\mathbb{1}[t-LT_i \ge 1] - d_{i,t} \qquad \forall i \in I,\ t \in T
$$

**C2 — Quantité minimale de commande (MOQ)**, liée à la variable binaire $z_{i,t}$ :
$$
MOQ_i \cdot z_{i,t} \;\le\; O_{i,t} \;\le\; M \cdot z_{i,t} \qquad \forall i \in I,\ t \in T
$$

**C3 — Stock de sécurité**, à partir de $t=2$ (le stock initial est une donnée subie, pas une
décision — l'exiger dès $t=1$ rendrait le modèle infaisable par construction) :
$$
S_{i,t} \ge SS_i \qquad \forall i \in I,\ t \in T,\ t \ge 2
$$

**C4 — Fermeture fournisseur en août** : aucune commande ne peut être **passée** pendant les
périodes de $A$ (elle peut en revanche être **reçue**, si passée avant) :
$$
O_{i,t} = 0 \qquad \forall i \in I,\ t \in A
$$

**C5 — Capacité de stockage** :
$$
\sum_{i \in I} v_i \, S_{i,t} \;\le\; V_{max} \qquad \forall t \in T
$$

**C6 — Cible de valeur de stock en fin d'horizon** ($T_{max}=5$, Septembre) :
$$
\sum_{i \in I} p_i \, S_{i,T_{max}} \;\le\; TARGET
$$


In [ ]:
from pyomo.environ import *
from IPython.display import display
import pandas as pd
import numpy as np
import math


---
## 3. Implémentation — mêmes 5 étapes que la démarche AMPL/GUSEK

### 1. LES PARAMÈTRES


In [ ]:
model = AbstractModel()

model.T   = Param(within=PositiveIntegers)
model.PER = RangeSet(1, model.T)
model.AoutSet = Set(within=model.PER)   # C4

model.I = Set()
model.M = Param()

model.prix          = Param(model.I)   # p_i
model.stock_init    = Param(model.I)   # S0_i
model.LT            = Param(model.I)
model.MOQ           = Param(model.I)
model.volume        = Param(model.I)   # v_i
model.SS            = Param(model.I)
model.cout_stockage = Param(model.I)   # c^s_i
model.cout_commande = Param(model.I)   # c^o_i

model.demande = Param(model.I, model.PER)

model.TARGET = Param()
model.V_max  = Param()


### 2. LES VARIABLES

In [ ]:
model.S = Var(model.I, model.PER, domain=NonNegativeReals)   # S_{i,t}
model.O = Var(model.I, model.PER, domain=NonNegativeReals)   # O_{i,t}
model.z = Var(model.I, model.PER, domain=Binary)              # z_{i,t}


### 3. LA FONCTION OBJECTIF (et les contraintes C1 → C6)

In [ ]:
def cost_rule(m):
    return (sum(m.cout_stockage[i]*m.S[i,t] for i in m.I for t in m.PER)
          + sum(m.cout_commande[i]*m.z[i,t]  for i in m.I for t in m.PER))
model.cost = Objective(rule=cost_rule, sense=minimize)


In [ ]:
# C1 — Bilan de stock
def c1_rule(m, i, t):
    S_prev = m.stock_init[i] if t == 1 else m.S[i, t-1]
    t_cmd  = t - m.LT[i]
    reception = m.O[i, t_cmd] if t_cmd >= 1 else 0
    return m.S[i,t] == S_prev + reception - m.demande[i,t]
model.C1_Bilan = Constraint(model.I, model.PER, rule=c1_rule)

# C2 — MOQ
def c2a_rule(m, i, t):
    return m.O[i,t] >= m.MOQ[i] * m.z[i,t]
model.C2a_MOQ_min = Constraint(model.I, model.PER, rule=c2a_rule)

def c2b_rule(m, i, t):
    return m.O[i,t] <= m.M * m.z[i,t]
model.C2b_MOQ_max = Constraint(model.I, model.PER, rule=c2b_rule)

# C3 — Stock de sécurité, à partir de t=2
def c3_rule(m, i, t):
    if t == 1:
        return Constraint.Skip
    return m.S[i,t] >= m.SS[i]
model.C3_StockSecurite = Constraint(model.I, model.PER, rule=c3_rule)

# C4 — Fermeture fournisseur en août
def c4_rule(m, i, t):
    return m.O[i,t] == 0
model.C4_FermetureFournisseur = Constraint(model.I, model.AoutSet, rule=c4_rule)

# C5 — Capacité de stockage
def c5_rule(m, t):
    return sum(m.volume[i]*m.S[i,t] for i in m.I) <= m.V_max
model.C5_CapaciteStockage = Constraint(model.PER, rule=c5_rule)

# C6 — Cible de valeur de stock en fin d'horizon
def c6_rule(m):
    return sum(m.prix[i]*m.S[i, m.T] for i in m.I) <= m.TARGET
model.C6_CibleFiscale = Constraint(rule=c6_rule)


### 4. SOLVE

In [ ]:
def resoudre(instance, solveur="cbc", verbose=True):
    opt = SolverFactory(solveur)
    return opt.solve(instance, tee=verbose)


In [ ]:
import shutil
chemin_cbc = shutil.which("cbc")
if chemin_cbc is None:
    import subprocess
    subprocess.run(["apt-get", "install", "-y", "coinor-cbc"], check=True)
    chemin_cbc = shutil.which("cbc")
assert chemin_cbc is not None, "CBC introuvable"
print(f"CBC disponible : {chemin_cbc}")


### 5. METTRE EN PLACE LA DATA

Six références, saisies directement dans le notebook. Le stock initial et les prix sont
calibrés pour que la valeur totale de départ soit d'environ 4 000 000 €.


In [ ]:
noms_mois = {1:"Mai", 2:"Juin", 3:"Juillet", 4:"Août", 5:"Septembre"}
T_max  = 5
t_aout = [4]

references = {
    "PN-A": {"prix": 150, "LT": 1, "MOQ": 100,  "cout_commande": 200, "volume": 0.05},
    "PN-B": {"prix": 60,  "LT": 1, "MOQ": 500,  "cout_commande": 150, "volume": 0.03},
    "PN-C": {"prix": 400, "LT": 1, "MOQ": 150,  "cout_commande": 400, "volume": 0.08},
    "PN-D": {"prix": 25,  "LT": 1, "MOQ": 1000, "cout_commande": 100, "volume": 0.01},
    "PN-E": {"prix": 120, "LT": 2, "MOQ": 300,  "cout_commande": 250, "volume": 0.04},
    "PN-F": {"prix": 300, "LT": 2, "MOQ": 150,  "cout_commande": 300, "volume": 0.06},
}
refs = list(references.keys())

stock_init = {"PN-A": 3000, "PN-B": 10000, "PN-C": 2000, "PN-D": 20000, "PN-E": 7000, "PN-F": 3000}

valeur_initiale = sum(stock_init[i]*references[i]["prix"] for i in refs)
print(f"Valeur de stock initiale (mai) : {valeur_initiale:,.0f} €")

TARGET = 2_000_000
print(f"Cible fiscale (fin septembre)  : {TARGET:,.0f} €")


In [ ]:
# Demande mensuelle : taux de consommation ~10.5%/mois du stock initial, légère saisonnalité
taux_conso = 0.105
variation  = {1: 0.95, 2: 1.00, 3: 1.05, 4: 1.00, 5: 1.00}

demande = {}
for i in refs:
    base = round(taux_conso * stock_init[i])
    for t in range(1, T_max + 1):
        demande[(i, t)] = round(base * variation[t])

df_demande = pd.DataFrame({i: [demande[(i,t)] for t in range(1,T_max+1)] for i in refs},
                           index=[noms_mois[t] for t in range(1,T_max+1)]).T
print("Demande mensuelle par référence :")
display(df_demande)


**Stock de sécurité** (formule EOQ classique) :
$$SS_i = z_\alpha \cdot \sigma_{D,i} \cdot \sqrt{LT_i}$$


In [ ]:
z_alpha = 1.65   # niveau de service ~95%
sigma_D = {i: np.std([demande[(i,t)] for t in range(1, T_max+1)]) for i in refs}
SS = {i: round(z_alpha * sigma_D[i] * math.sqrt(references[i]["LT"])) for i in refs}

df_ss = pd.DataFrame({"Stock initial": stock_init, "SS": SS,
                       "LT (mois)": {i: references[i]["LT"] for i in refs}})
display(df_ss)


**Capacité de stockage** : fixée à 1,25× le volume nécessaire au stock initial (le volume
ne peut que décroître ensuite, la contrainte C5 reste donc valide sur tout l'horizon).


In [ ]:
volume_initial = sum(stock_init[i]*references[i]["volume"] for i in refs)
V_max = round(volume_initial * 1.25)
print(f"Volume de stock initial : {volume_initial:,.0f} m³")
print(f"Capacité de stockage (V_max) : {V_max:,.0f} m³")

cout_stockage = {i: round(0.20 * references[i]["prix"] / 12, 4) for i in refs}   # tau=20%/an
M_bigM = max(references[i]["MOQ"] for i in refs) * 20


In [ ]:
data = DataPortal()
data['T'] = {None: T_max}
data['AoutSet'] = t_aout
data['I'] = refs
data['M'] = {None: M_bigM}

data['prix']          = {i: references[i]["prix"] for i in refs}
data['stock_init']    = stock_init
data['LT']            = {i: references[i]["LT"] for i in refs}
data['MOQ']           = {i: references[i]["MOQ"] for i in refs}
data['volume']        = {i: references[i]["volume"] for i in refs}
data['SS']            = SS
data['cout_stockage'] = cout_stockage
data['cout_commande'] = {i: references[i]["cout_commande"] for i in refs}
data['demande']       = demande
data['TARGET']        = {None: TARGET}
data['V_max']         = {None: V_max}

instance = model.create_instance(data)
print("Instance créée :", len(refs), "références,", T_max, "mois")


### Résolution

In [ ]:
resultats = resoudre(instance, solveur="cbc", verbose=True)
print("\nStatut :", resultats.solver.termination_condition)
print("Coût total optimal :", round(value(instance.cost), 2), "€")


### Résultats

In [ ]:
valeur_finale = sum(references[i]["prix"]*value(instance.S[i, T_max]) for i in refs)
print(f"Valeur de stock initiale : {valeur_initiale:,.0f} €")
print(f"Valeur de stock finale (Septembre) : {valeur_finale:,.0f} €")
print(f"Cible : {TARGET:,.0f} €")
print("Cible respectée" if valeur_finale <= TARGET else "Cible dépassée")

lignes = []
for i in refs:
    for t in range(1, T_max+1):
        lignes.append({
            "PN": i, "Mois": noms_mois[t],
            "Demande": demande[(i,t)],
            "Commande (O)": round(value(instance.O[i,t])),
            "Stock fin de mois (S)": round(value(instance.S[i,t])),
            "z (commande passée)": int(round(value(instance.z[i,t]))),
        })
df_resultats = pd.DataFrame(lignes)
for i in refs:
    print(f"\n--- {i} ---")
    display(df_resultats[df_resultats["PN"]==i].drop(columns="PN").set_index("Mois"))


In [ ]:
import matplotlib.pyplot as plt

valeurs_mois = []
for t in range(1, T_max+1):
    v = sum(references[i]["prix"]*value(instance.S[i,t]) for i in refs)
    valeurs_mois.append(v)

fig, ax = plt.subplots(figsize=(9,5))
ax.plot([noms_mois[t] for t in range(1,T_max+1)], valeurs_mois, marker="o", linewidth=2.5, label="Valeur de stock")
ax.axhline(TARGET, color="red", linestyle="--", label=f"Cible ({TARGET:,.0f} €)")
ax.axhline(valeur_initiale, color="gray", linestyle=":", label=f"Valeur initiale ({valeur_initiale:,.0f} €)")
ax.set_title("Trajectoire de la valeur de stock (Mai -> Septembre)")
ax.set_xlabel("Mois"); ax.set_ylabel("Valeur du stock (€)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
